# Artist Clustering using Gower Distance + K-Medoids

Aim: Grouping Artists based on avg_position, position_stddev and Genre.

In [3]:
import duckdb
import pandas as pd

con = duckdb.connect('../../data/warehouse.duckdb')
df = con.sql("SELECT * FROM fct_artist_features").df()
df.head()

,artist_id,artist_name,term_id,snapshot_month,avg_position,position_stddev,position_trend_slope,snapshot_count,genres
0,4tLIGM52ee6vMMdBtDZFi7,Faderhead,l,2026-01-01,12.625000,0.517549,1.907525e-06,8,"[synthpop, ebm, darkwave, industrial, industri..."
1,7cWnPCnFKYKg5W0JKaf118,ASP,l,2026-01-01,3.000000,0.000000,0.000000e+00,8,"[gothic rock, darkwave, medieval metal, mediev..."
2,1Jgp0YCPHCJx5XD7nlfGVN,Aesthetic Perfection,l,2026-02-01,9.000000,0.000000,0.000000e+00,35,"[darkwave, industrial metal, industrial, indus..."
3,7LhCMNjTNM9R2a0fajH9l8,Das Ich,l,2026-02-01,10.000000,0.000000,0.000000e+00,35,"[darkwave, industrial, industrial metal, ebm, ..."
4,4N99bzdRBNRi7hGNCpMuhu,Suicide Commando,l,2026-02-01,19.657143,0.481594,-6.530029e-07,35,"[darkwave, industrial, industrial metal, ebm, ..."


In [4]:
df.tail()

,artist_id,artist_name,term_id,snapshot_month,avg_position,position_stddev,position_trend_slope,snapshot_count,genres
410,5Ugr8SZik8bMfgldDbJecL,Noisuf-X,m,2026-05-01,20.000000,NaN,NaN,1,"[ebm, industrial rock, industrial metal, indus..."
411,7rH8vpDNenEoyRm1NDJWzW,X-RX,l,2026-05-01,20.000000,NaN,NaN,1,"[darkwave, industrial metal, industrial, ebm]"
412,2VwHIKqBOx8TfwVy8tU2Ya,Black Heaven,m,2026-07-01,39.666667,1.154701,-8.154937e-07,3,"[gothic rock, darkwave, industrial, synthpop, ..."
413,1wGcs6i2IJbgXxE3bDIm9k,Nachtmahr,s,2026-07-01,2.000000,0.000000,0.000000e+00,4,"[darkwave, industrial metal, industrial, ebm]"
414,6VMsDolxD0gqjdLadDpUzj,Eisenfunk,m,2026-07-01,18.000000,NaN,NaN,1,"[ebm, industrial metal, industrial]"


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 415 entries, 0 to 414
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   artist_id             415 non-null    str           
 1   artist_name           415 non-null    str           
 2   term_id               415 non-null    str           
 3   snapshot_month        415 non-null    datetime64[us]
 4   avg_position          415 non-null    float64       
 5   position_stddev       327 non-null    float64       
 6   position_trend_slope  327 non-null    float64       
 7   snapshot_count        415 non-null    int64         
 8   genres                413 non-null    object        
dtypes: datetime64[us](1), float64(3), int64(1), object(1), str(3)
memory usage: 29.3+ KB


In [6]:
df.describe()

,snapshot_month,avg_position,position_stddev,position_trend_slope,snapshot_count
count,415,415.000000,327.000000,3.270000e+02,415.000000
mean,2026-04-12 14:48:17.349397,16.023742,1.032333,1.580840e-07,7.667470
min,2026-01-01 00:00:00,1.000000,0.000000,-1.607193e-05,1.000000
25%,2026-02-01 00:00:00,6.612500,0.000000,-8.035333e-08,2.000000
50%,2026-05-01 00:00:00,13.000000,0.000000,0.000000e+00,3.000000
75%,2026-07-01 00:00:00,20.000000,1.154701,0.000000e+00,8.000000
max,2026-07-01 00:00:00,50.000000,16.970563,2.192040e-05,35.000000
std,NaN,12.464801,1.971606,3.644273e-06,11.071366


In [8]:
df['genres'].apply(lambda g: len(g) if isinstance(g, list) else 0).value_counts()

genres
0    415
Name: count, dtype: int64

In [11]:
import numpy as np

df['genres'].apply(lambda g: len(g) if isinstance(g, (list, np.ndarray)) else 0).value_counts()

genres
4    129
5     74
8     55
7     49
6     44
2     39
1     16
3      6
0      2
9      1
Name: count, dtype: int64

With only two artists not having a genre, with dropping them we will nbot loose much data.

In [12]:
df_clean = df[df['genres'].apply(lambda g: isinstance(g, (list, np.ndarray)))].copy()
print(f"Removed {len(df) - len(df_clean)} rows without genre data")
df_clean.shape

Removed 2 rows without genre data


(413, 9)

In [13]:
features = df_clean[['artist_id', 'artist_name', 'avg_position', 'position_stddev', 'genres']].copy()
features['position_stddev'] = features['position_stddev'].fillna(0)
features.info()

<class 'pandas.DataFrame'>
Index: 413 entries, 0 to 414
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   artist_id        413 non-null    str    
 1   artist_name      413 non-null    str    
 2   avg_position     413 non-null    float64
 3   position_stddev  413 non-null    float64
 4   genres           413 non-null    object 
dtypes: float64(2), object(1), str(2)
memory usage: 19.4+ KB


In [14]:
gower_input = features[['avg_position', 'position_stddev']].copy()
gower_input['primary_genre'] = features['genres'].apply(lambda g: g[0])
gower_input.head()

,avg_position,position_stddev,primary_genre
0,12.625000,0.517549,synthpop
1,3.000000,0.000000,gothic rock
2,9.000000,0.000000,darkwave
3,10.000000,0.000000,darkwave
4,19.657143,0.481594,darkwave


In [16]:
gower_input['primary_genre'] = gower_input['primary_genre'].astype('object')
gower_input.dtypes

avg_position       float64
position_stddev    float64
primary_genre       object
dtype: object

In [17]:
distance_matrix = gower.gower_matrix(gower_input)
distance_matrix.shape

(413, 413)

silhouette coefficient calculation to get the best k-value.

In [19]:
import kmedoids
from sklearn.metrics import silhouette_score

scores = {}
for k in range(2, 10):
    result = kmedoids.fasterpam(distance_matrix, k, random_state=42)
    labels = result.labels
    scores[k] = silhouette_score(distance_matrix, labels, metric='precomputed')

scores

{2: 0.20920711755752563,
 3: 0.237661212682724,
 4: 0.3485890030860901,
 5: 0.4033263027667999,
 6: 0.4694613218307495,
 7: 0.5324541330337524,
 8: 0.5740538835525513,
 9: 0.4779021739959717}

seems to be 8, but on the other hand it is rising untill neary the end. We need to validate if the peak arises lalter.

In [20]:
scores_extended = {}
for k in range(2, 15):
    result = kmedoids.fasterpam(distance_matrix, k, random_state=42)
    labels = result.labels
    scores_extended[k] = silhouette_score(distance_matrix, labels, metric='precomputed')

scores_extended

{2: 0.20920711755752563,
 3: 0.237661212682724,
 4: 0.3485890030860901,
 5: 0.4033263027667999,
 6: 0.4694613218307495,
 7: 0.5324541330337524,
 8: 0.5740538835525513,
 9: 0.4779021739959717,
 10: 0.5165457725524902,
 11: 0.5408990979194641,
 12: 0.5548331141471863,
 13: 0.5767327547073364,
 14: 0.5777465105056763}

Still does not provide a realy meaningfull answer, since it is dropping after 8, but then rising again. Lets take a look at the actual distribution.

In [21]:
result_8 = kmedoids.fasterpam(distance_matrix, 8, random_state=42)
import pandas as pd
pd.Series(result_8.labels).value_counts().sort_index()

0     24
1     50
2     60
3     16
4     37
5     45
6    117
7     64
Name: count, dtype: int64

Not bad, but also not extremly good. Let's take a look at the marginal gain, instead of only the global maximum.

In [22]:
ks = sorted(scores_extended.keys())
gains = {ks[i]: scores_extended[ks[i]] - scores_extended[ks[i-1]] for i in range(1, len(ks))}
gains

{3: 0.028454095125198364,
 4: 0.11092779040336609,
 5: 0.05473729968070984,
 6: 0.06613501906394958,
 7: 0.06299281120300293,
 8: 0.04159975051879883,
 9: -0.09615170955657959,
 10: 0.038643598556518555,
 11: 0.024353325366973877,
 12: 0.013934016227722168,
 13: 0.021899640560150146,
 14: 0.0010137557983398438}

8 still seems to be the best option.

In [23]:
final_result = kmedoids.fasterpam(distance_matrix, 8, random_state=42)
features['cluster'] = final_result.labels
features['cluster'].value_counts().sort_index()

cluster
0     24
1     50
2     60
3     16
4     37
5     45
6    117
7     64
Name: count, dtype: int64

In [24]:
features.groupby('cluster')[['avg_position', 'position_stddev']].mean()

,avg_position,position_stddev
cluster,,
0,16.754586,0.840841
1,19.728578,0.784548
2,9.750105,1.126344
3,19.875656,0.724311
4,5.569361,0.418831
5,31.030291,0.720711
6,15.220670,0.666928
7,14.235147,1.141195


In [26]:
for cluster_id in sorted(features['cluster'].unique()):
    top_genres = features[features['cluster'] == cluster_id]['primary_genre'].value_counts().head(3)
    print(f"Cluster {cluster_id}:")
    print(top_genres)
    print()

Cluster 0:
primary_genre
synthpop     23
deathrock     1
Name: count, dtype: int64

Cluster 1:
primary_genre
industrial     45
black metal     1
neofolk         1
Name: count, dtype: int64

Cluster 2:
primary_genre
gothic rock    52
deathrock       2
black metal     2
Name: count, dtype: int64

Cluster 3:
primary_genre
industrial metal    15
neoclassical         1
Name: count, dtype: int64

Cluster 4:
primary_genre
gothic metal      28
medieval metal     5
black metal        1
Name: count, dtype: int64

Cluster 5:
primary_genre
christmas      20
neofolk         4
black metal     4
Name: count, dtype: int64

Cluster 6:
primary_genre
darkwave        114
deathrock         2
neoclassical      1
Name: count, dtype: int64

Cluster 7:
primary_genre
ebm                 57
children's music     3
black metal          2
Name: count, dtype: int64



The clustering does not bring any additional value or knowledge, but basically recreates .groupby('primary_genre').

In [27]:
all_genres = set()
for g_list in features['genres']:
    all_genres.update(g_list)
sorted(all_genres)

['adult standards',
 'ambient folk',
 'avant-garde',
 'big band',
 'black metal',
 "children's music",
 'choral',
 'christmas',
 'cold wave',
 'dark ambient',
 'darkwave',
 'deathrock',
 'doom metal',
 'east coast hip hop',
 'ebm',
 'electropop',
 'finnish rock',
 'folk',
 'gothic metal',
 'gothic rock',
 'hip hop',
 'horror punk',
 'industrial',
 'industrial metal',
 'industrial rock',
 'jazz',
 'krautrock',
 'medieval',
 'medieval metal',
 'metal',
 'neoclassical',
 'neofolk',
 'new wave',
 'post-punk',
 'rap',
 'swing music',
 'symphonic metal',
 'synthpop',
 'synthwave',
 'vocal jazz']

Since we have a full list of genres per artist anyway (intentionally!), we can use the whole list for the clustering. But gower-distance cannot be calculated based on a list of categorical feates, but only on single ones. So we are using Jaccard-distance for the categorical features and gower for the numerical ones and combine them.

In [28]:
import numpy as np

def jaccard_distance(set_a, set_b):
    a, b = set(set_a), set(set_b)
    if not a and not b:
        return 0.0
    return 1 - len(a & b) / len(a | b)

genre_lists = features['genres'].apply(list).tolist()
n = len(genre_lists)
jaccard_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(i+1, n):
        d = jaccard_distance(genre_lists[i], genre_lists[j])
        jaccard_matrix[i, j] = d
        jaccard_matrix[j, i] = d

In [30]:
jaccard_matrix.shape

(413, 413)

In [29]:
gower_numeric = gower.gower_matrix(features[['avg_position', 'position_stddev']])

In [31]:
gower_numeric.shape

(413, 413)

In [32]:
combined_distance = 0.5 * gower_numeric + 0.5 * jaccard_matrix

In [33]:
scores_combined = {}
for k in range(2, 15):
    result = kmedoids.fasterpam(combined_distance, k, random_state=42)
    labels = result.labels
    scores_combined[k] = silhouette_score(combined_distance, labels, metric='precomputed')

scores_combined

{2: 0.15546617514646496,
 3: 0.2001377359012476,
 4: 0.2453687614640746,
 5: 0.2598809287357286,
 6: 0.2949418263159598,
 7: 0.28046093427980245,
 8: 0.3480237026109342,
 9: 0.37250029298381615,
 10: 0.40135909270085907,
 11: 0.40497443140474515,
 12: 0.3979120323709125,
 13: 0.4058603184439385,
 14: 0.40735198235974723}

In [34]:
ks = sorted(scores_combined.keys())
gains_combined = {ks[i]: scores_combined[ks[i]] - scores_combined[ks[i-1]] for i in range(1, len(ks))}
gains_combined

{3: 0.044671560754782624,
 4: 0.045231025562827015,
 5: 0.014512167271653997,
 6: 0.035060897580231176,
 7: -0.014480892036157322,
 8: 0.06756276833113173,
 9: 0.024476590372881968,
 10: 0.028858799717042916,
 11: 0.003615338703886084,
 12: -0.007062399033832656,
 13: 0.007948286073026,
 14: 0.0014916639158087386}

We will take k=8 again, because of the biggest gain.

In [35]:
final_result_combined = kmedoids.fasterpam(combined_distance, 8, random_state=42)
features['cluster_v2'] = final_result_combined.labels
features['cluster_v2'].value_counts().sort_index()

cluster_v2
0    53
1    31
2    63
3    48
4    25
5    85
6    28
7    80
Name: count, dtype: int64

In [36]:
features.groupby('cluster_v2')[['avg_position', 'position_stddev']].mean()

,avg_position,position_stddev
cluster_v2,,
0,13.784263,0.957372
1,15.801449,0.548608
2,13.672681,1.198512
3,11.187779,0.947838
4,4.482273,0.434475
5,20.486026,0.675411
6,30.029762,1.329394
7,15.910593,0.541601


In [ ]:
for cluster_id in sorted(features['cluster_v2'].unique()):
    genres_in_cluster = []
    for g_list in features[features['cluster_v2'] == cluster_id]['genres']:
        genres_in_cluster.extend(g_list)
    print(f"Cluster {cluster_id} (n={sum(features['cluster_v2'] == cluster_id)}):")
    print(pd.Series(genres_in_cluster).value_counts().head(5))
    print()

Cluster 0 (n=53):
deathrock      50
gothic rock    50
darkwave       50
industrial     46
ebm            44
Name: count, dtype: int64

Cluster 1 (n=31):
children's music    21
christmas           19
medieval metal       8
choral               2
big band             1
Name: count, dtype: int64

Cluster 2 (n=63):
ebm                 62
industrial          62
industrial metal    60
darkwave            56
synthpop             5
Name: count, dtype: int64

Cluster 3 (n=48):
darkwave          48
medieval metal    48
ebm               48
industrial        47
gothic rock       40
Name: count, dtype: int64

Cluster 4 (n=25):
gothic metal       24
gothic rock        24
darkwave           23
symphonic metal    17
deathrock           7
Name: count, dtype: int64

Cluster 5 (n=85):
ebm            84
industrial     83
darkwave       82
synthpop       80
gothic rock    18
Name: count, dtype: int64

Cluster 6 (n=28):
neofolk         18
neoclassical    15
ambient folk     7
darkwave         5
dark ambien

: 

In [ ]:
df['genres'].apply(lambda g: len(g) if isinstance(g, list) else 0).value_counts()

genres
0    415
Name: count, dtype: int64

This seems to give a better result. cluster 0 is still dominated by most popular genres, but gives a more diverse result. And cluster 1 does not show similar music, but music that I listen to, in the some context (with the kids).